In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('UCI_Credit_Card.csv')
df.head(2)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1


In [3]:
df.drop('ID',axis=1,inplace=True) #dropping insignificant column

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   LIMIT_BAL                   30000 non-null  float64
 1   SEX                         30000 non-null  int64  
 2   EDUCATION                   30000 non-null  int64  
 3   MARRIAGE                    30000 non-null  int64  
 4   AGE                         30000 non-null  int64  
 5   PAY_0                       30000 non-null  int64  
 6   PAY_2                       30000 non-null  int64  
 7   PAY_3                       30000 non-null  int64  
 8   PAY_4                       30000 non-null  int64  
 9   PAY_5                       30000 non-null  int64  
 10  PAY_6                       30000 non-null  int64  
 11  BILL_AMT1                   30000 non-null  float64
 12  BILL_AMT2                   30000 non-null  float64
 13  BILL_AMT3                   300

In [5]:
for col in ['SEX','EDUCATION','MARRIAGE','PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']:
    df[col]=df[col].astype('category')

In [6]:
x=df.drop('default.payment.next.month',axis=1) #Independent Features
y=df['default.payment.next.month']  #Dependent Featur

In [7]:
cat_features = x.select_dtypes(include=['object', 'category']).columns.tolist()
num_features1 = x.select_dtypes(include=['int64', 'float64']).columns.tolist()

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
         ("OneHotEncoder", oh_transformer, cat_features),  #Applying one-hot encoding to categorical columns
          ("StandardScaler", numeric_transformer, num_features1) #Applying StandardScaler to numeric columns
    ]
)

In [8]:
x_scaled=preprocessor.fit_transform(x) #Scaling the data

In [9]:
!pip install xgboost

In [10]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import precision_score,accuracy_score,recall_score,f1_score,roc_auc_score

In [11]:
#Preparing the training and testing dataset 
x_train,x_test,y_train,y_test=train_test_split(x_scaled,y,test_size=0.2,random_state=42)
x_train.shape,y_test.shape

((24000, 82), (6000,))

In [12]:
xg=XGBClassifier()
xg.fit(x_train,y_train)
y_pred=xg.predict(x_test)
model_test_accuracy = accuracy_score(y_test, y_pred) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.8152
F1 score:0.7950
Precision:0.6367
Recall:0.3618
ROC:0.6520


In [14]:
!pip install imblearn


   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ---------------------------------------- 3/3 [imblearn]



XGBOOST WITH SMOTE

In [15]:
#We don't use SMOTE-NC here as we have already encoded the categorical features using OHE
from imblearn.over_sampling import SMOTE 
sm=SMOTE(random_state=42)

In [16]:
x_new,y_new=sm.fit_resample(x_train,y_train) #Applying SMOTE

# Define and fit the Random Forest Classifier
model1=XGBClassifier()
model1.fit(x_new,y_new)

y_pred_smote=model1.predict(x_test)  # Make predictions on the test set
model_test_accuracy = accuracy_score(y_test, y_pred_smote) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred_smote, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred_smote) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred_smote) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred_smote) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.8092
F1 score:0.7949
Precision:0.5946
Recall:0.4021
ROC:0.6627


XGBOOST WITH UNDER SAMPLING

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(x_train, y_train)

# Define and fit the Random Forest Classifier
model = XGBClassifier()
model.fit(X_rus, y_rus)

# Make predictions on the test set
y_pred = model.predict(x_test)

# Calculate metrics on the test set
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
auc = roc_auc_score(y_test, model.predict_proba(x_test)[:, 1])

# Print metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")
print(f"AUC: {auc:.4f}")

Accuracy: 0.7157
Precision: 0.4077
F1 Score: 0.5044
Recall: 0.6611
AUC: 0.7634


In [ ]:
#Recall increases after under sampling but precision and F1 score decreases drastically